In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -------------------------
# Tables
# -------------------------
bronze_tbl = "capstone.bronze.pharmacies"
geo_tbl = "capstone.silver.geography"
silver_tbl = "capstone.silver.pharmacy"

# -------------------------
# 1. Read bronze pharmacy
# -------------------------
df = spark.table(bronze_tbl)

# -------------------------
# 2. Select and rename needed columns
# -------------------------
df = df.select(
    "pharmacy_id",
    "pharmacy_name",
    F.col("address_raw").alias("address"),
    F.col("city_raw").alias("city"),
    F.col("county_raw").alias("county"),
    F.col("state_raw").alias("state"),
    F.col("latitude_raw").alias("latitude"),
    F.col("longitude_raw").alias("longitude"),
    "ingestion_timestamp",
    "source_system"
)

# -------------------------
# 3. Basic cleaning
# -------------------------
df = (
    df.withColumn("pharmacy_id", F.trim(F.col("pharmacy_id")))
      .withColumn("pharmacy_name", F.initcap(F.trim(F.col("pharmacy_name"))))
      .withColumn("address", F.trim(F.col("address")))
      .withColumn("city", F.initcap(F.trim(F.col("city"))))
      .withColumn("county", F.initcap(F.trim(F.col("county"))))
      .withColumn("state", F.upper(F.trim(F.col("state"))))
      .withColumn("source_system", F.upper(F.trim(F.col("source_system"))))
)

# -------------------------
# 4. Safe coordinate parsing
#    Handles values like:
#    44.9778N, 93.2650W, etc.
# -------------------------
df = (
    df.withColumn(
        "latitude",
        F.expr("try_cast(regexp_replace(latitude, '[^0-9.-]', '') AS double)")
    )
    .withColumn(
        "longitude",
        F.expr("try_cast(regexp_replace(longitude, '[^0-9.-]', '') AS double)")
    )
)

# -------------------------
# 5. Fix longitude sign for W values if needed
# -------------------------
df = df.withColumn(
    "longitude",
    F.when(
        F.col("longitude").isNotNull() &
        (F.upper(F.col("state")).isin(
            "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA",
            "KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ",
            "NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT",
            "VA","WA","WV","WI","WY","DC"
        )) &
        (F.col("longitude") > 0),
        -F.col("longitude")
    ).otherwise(F.col("longitude"))
)

# -------------------------
# 6. Remove rows with null PK
# -------------------------
df = df.filter(F.col("pharmacy_id").isNotNull() & (F.col("pharmacy_id") != ""))

# -------------------------
# 7. Read geography for region lookup
# -------------------------
geo_df = (
    spark.table(geo_tbl)
    .select(
        F.initcap(F.trim(F.col("city"))).alias("geo_city"),
        F.initcap(F.trim(F.col("county"))).alias("geo_county"),
        F.upper(F.trim(F.col("state"))).alias("geo_state"),
        F.trim(F.col("geo_id")).alias("geo_id"),
        F.trim(F.col("region")).alias("region")
    )
)

# -------------------------
# 8. Join pharmacy with geography
#    to get geo_id + region
# -------------------------
df = (
    df.join(
        geo_df,
        (df.city == geo_df.geo_city) &
        (df.county == geo_df.geo_county) &
        (df.state == geo_df.geo_state),
        "left"
    )
    .select(
        df.pharmacy_id,
        df.pharmacy_name,
        df.address,
        df.city,
        df.county,
        df.state,
        F.col("region"),
        F.col("geo_id"),
        df.latitude,
        df.longitude,
        df.ingestion_timestamp,
        df.source_system
    )
)

# -------------------------
# 9. Deduplication
#    Keep latest record per pharmacy_id
# -------------------------
window_spec = Window.partitionBy("pharmacy_id").orderBy(F.col("ingestion_timestamp").desc())

df = (
    df.withColumn("row_num", F.row_number().over(window_spec))
      .filter(F.col("row_num") == 1)
      .drop("row_num")
)

# -------------------------
# 10. Write to Silver
# -------------------------
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_tbl)

In [0]:
%sql
-- ===============================
-- 1. Total Rows
-- ===============================
SELECT 
'Total Rows' AS check_type,
COUNT(*) AS result
FROM capstone.silver.pharmacy

UNION ALL

-- ===============================
-- 2. Missing Pharmacy ID
-- ===============================
SELECT
'Missing Pharmacy ID',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE pharmacy_id IS NULL

UNION ALL

-- ===============================
-- 3. Missing Pharmacy Name
-- ===============================
SELECT
'Missing Pharmacy Name',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE pharmacy_name IS NULL OR TRIM(pharmacy_name) = ''

UNION ALL

-- ===============================
-- 4. Missing Address
-- ===============================
SELECT
'Missing Address',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE address IS NULL OR TRIM(address) = ''

UNION ALL

-- ===============================
-- 5. Missing City
-- ===============================
SELECT
'Missing City',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE city IS NULL OR TRIM(city) = ''

UNION ALL

-- ===============================
-- 6. Missing County
-- ===============================
SELECT
'Missing County',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE county IS NULL OR TRIM(county) = ''

UNION ALL

-- ===============================
-- 7. Missing State
-- ===============================
SELECT
'Missing State',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE state IS NULL OR TRIM(state) = ''

UNION ALL

-- ===============================
-- 8. Missing Region
-- ===============================
SELECT
'Missing Region',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE region IS NULL OR TRIM(region) = ''

UNION ALL

-- ===============================
-- 9. Missing Geo ID
-- ===============================
SELECT
'Missing Geo ID',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE geo_id IS NULL OR TRIM(geo_id) = ''

UNION ALL

-- ===============================
-- 10. Invalid Latitude
-- ===============================
SELECT
'Invalid Latitude',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE latitude IS NOT NULL
  AND (latitude < -90 OR latitude > 90)

UNION ALL

-- ===============================
-- 11. Invalid Longitude
-- ===============================
SELECT
'Invalid Longitude',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE longitude IS NOT NULL
  AND (longitude < -180 OR longitude > 180)

UNION ALL

-- ===============================
-- 12. Duplicate Pharmacy IDs
-- ===============================
SELECT
'Duplicate Pharmacy IDs',
COUNT(*)
FROM (
    SELECT pharmacy_id
    FROM capstone.silver.pharmacy
    GROUP BY pharmacy_id
    HAVING COUNT(*) > 1
)

UNION ALL

-- ===============================
-- 13. Duplicate Pharmacy Records
-- ===============================
SELECT
'Duplicate Pharmacy Business Keys',
COUNT(*)
FROM (
    SELECT pharmacy_name, address, city, state
    FROM capstone.silver.pharmacy
    GROUP BY pharmacy_name, address, city, state
    HAVING COUNT(*) > 1
)

UNION ALL

-- ===============================
-- 14. Missing Ingestion Timestamp
-- ===============================
SELECT
'Missing Ingestion Timestamp',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE ingestion_timestamp IS NULL

UNION ALL

-- ===============================
-- 15. Missing Source System
-- ===============================
SELECT
'Missing Source System',
COUNT(*)
FROM capstone.silver.pharmacy
WHERE source_system IS NULL OR TRIM(source_system) = '';